In [2]:
import open3d as o3d
import numpy as np
import math
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt
import math
from functools import reduce

In [3]:
def load_point_clouds(voxel_size=0.0):
    pcds = []
    for i in range(1,3):
        pcd = o3d.io.read_point_cloud("data/%d_with_colormap.ply" %i)
        pcd.estimate_normals()
        pcd_down = pcd.voxel_down_sample(voxel_size=voxel_size)
        pcds.append(pcd_down)
    return pcds

In [4]:
voxel_size = 0.01
pcds_down = load_point_clouds(voxel_size)
o3d.visualization.draw_geometries(pcds_down)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [1]:
new_pcls = []
for pcd in pcds_down:
    print("A")
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=3)
    new_pcls.append(cl.select_by_index(ind))
o3d.visualization.draw_geometries(new_pcls)

NameError: name 'pcds_down' is not defined

In [5]:
def display_inlier_outlier(cloud, ind):
    inlier_cloud = cloud.select_by_index(ind)
    outlier_cloud = cloud.select_by_index(ind, invert=True)

    print("Showing outliers (red) and inliers (gray): ")
    outlier_cloud.paint_uniform_color([1, 0, 0])
    inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud],
                                      zoom=0.3412,
                                      front=[0.4257, -0.2125, -0.8795],
                                      lookat=[2.6172, 2.0475, 1.532],
                                      up=[-0.0694, -0.9768, 0.2024])


In [6]:
print("Statistical oulier removal")
cl, ind = pcds_down[1].remove_statistical_outlier(nb_neighbors=30,
                                                    std_ratio=3)
display_inlier_outlier(pcds_down[1], ind)


Statistical oulier removal
Showing outliers (red) and inliers (gray): 
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [7]:
def pairwise_registration(source, target, max_correspondence_distance_coarse, max_correspondence_distance_fine):
    print("Apply point-to-plane ICP")
    icp_coarse = o3d.pipelines.registration.registration_icp(
        source, target, max_correspondence_distance_coarse, np.identity(4),
        o3d.pipelines.registration.TransformationEstimationPointToPlane())
    icp_fine = o3d.pipelines.registration.registration_icp(
        source, target, max_correspondence_distance_fine,
        icp_coarse.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane())
    transformation_icp = icp_fine.transformation
    information_icp = o3d.pipelines.registration.get_information_matrix_from_point_clouds(
        source, target, max_correspondence_distance_fine,
        icp_fine.transformation)
    return transformation_icp, information_icp


def full_registration(pcds, max_correspondence_distance_coarse,
                      max_correspondence_distance_fine):
    pose_graph = o3d.pipelines.registration.PoseGraph()
    odometry = np.identity(4)
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(odometry))
    n_pcds = len(pcds)
    for source_id in range(n_pcds):
        for target_id in range(source_id + 1, n_pcds):
            transformation_icp, information_icp = pairwise_registration(
                pcds[source_id], pcds[target_id], max_correspondence_distance_coarse, max_correspondence_distance_fine)
            print("Build o3d.pipelines.registration.PoseGraph")
            if target_id == source_id + 1:  # odometry case
                odometry = np.dot(transformation_icp, odometry)
                pose_graph.nodes.append(
                    o3d.pipelines.registration.PoseGraphNode(
                        np.linalg.inv(odometry)))
                pose_graph.edges.append(
                    o3d.pipelines.registration.PoseGraphEdge(source_id,
                                                             target_id,
                                                             transformation_icp,
                                                             information_icp,
                                                             uncertain=False))
            else:  # loop closure case
                pose_graph.edges.append(
                    o3d.pipelines.registration.PoseGraphEdge(source_id,
                                                             target_id,
                                                             transformation_icp,
                                                             information_icp,
                                                             uncertain=True))
    return pose_graph

In [8]:
print("Full registration ...")
max_correspondence_distance_coarse = voxel_size * 7
max_correspondence_distance_fine = voxel_size * 0.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_down,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine)

Full registration ...
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.0746, RMSE 0.0401
[Open3D DEBUG] Residual : 9.51e-04 (# of elements : 7644)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.0759, RMSE 0.0402
[Open3D DEBUG] Residual : 9.59e-04 (# of elements : 7772)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.0774, RMSE 0.0403
[Open3D DEBUG] Residual : 9.61e-04 (# of elements : 7926)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.0788, RMSE 0.0405
[Open3D DEBUG] Residual : 9.60e-04 (# of elements : 8073)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.0798, RMSE 0.0405
[Open3D DEBUG] Residual : 9.45e-04 (# of elements : 8173)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.0808, RMSE 0.0408
[Open3D DEBUG] Residual : 9.29e-04 (# of elements : 8281)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.0818, RMSE 0.0411
[Open3D DEBUG] Residual : 9.40e-04 (# of elements : 8384)
[Open3D DEBUG] ICP Iteration #7: Fitness 0.0831, RMSE 0.0414
[Open3D DEBUG] Residual : 9.35e-04 (# of elements : 8516)
[

In [9]:
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)

Optimizing PoseGraph ...
[Open3D DEBUG] Validating PoseGraph - finished.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 2 nodes and 1 edges.
[Open3D DEBUG] Line process weight : 0.023850
[Open3D DEBUG] [Initial     ] residual : 1.090537e-29, lambda : 3.154633e-02
[Open3D DEBUG] Maximum coefficient of right term < 1.000000e-06
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 2 nodes and 1 edges.
[Open3D DEBUG] Line process weight : 0.023850
[Open3D DEBUG] [Initial     ] residual : 1.090537e-29, lambda : 3.154633e-02
[Open3D DEBUG] Maximum coefficient of right term < 1.000000e-06
[Open3D DEBUG] CompensateReferencePoseGraphNode : reference : 0


In [10]:
print("Transform points and display")
for point_id in range(len(pcds_down)):
    print(pose_graph.nodes[point_id].pose)
    pcds_down[point_id].transform(pose_graph.nodes[point_id].pose)
o3d.visualization.draw_geometries(pcds_down)



Transform points and display
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
[[ 0.94377802  0.02558212 -0.32958854 -0.63512753]
 [-0.042893    0.99804955 -0.0453573  -0.18022475]
 [ 0.32778536  0.05694427  0.94303452 -0.01371759]
 [ 0.          0.          0.          1.        ]]
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
